In [1]:
#create new file with perfect caratterization merging the manually check results and the automatic reseller results.
# Merge manually verified LPG points with attractiveness-weighted points
# and produce a combined GeoPackage with layers:
#   - resell
#   - filling
#   - resell_and_filling
#   - depot
# Manual points receive attractiveness = 1.
# automatic api reseller points are excluded if they fall inside a false point.
# Duplicates are removed based on geometry (manual first).
# All original columns are preserved.

In [ ]:
import geopandas as gpd
import pandas as pd

# Input / output files
DATA_DIR = "dataset_big"


manual_file = f"{DATA_DIR}/manual_check_nig_3857.gpkg"
attrac_file = f"{DATA_DIR}/lpg_points_nigeria_attractiveness.gpkg"
output_file = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"

# 1. Load manual layers
print("\n[1] Loading manual layers...")
manual_layers = {
    "reseller": "reseller",
    "filling": "filling",
    "primary_depot": "primary_depot",
    "false_points": "false_points"
}
manual_data = {}
for layer_name, layer_key in manual_layers.items():
    try:
        gdf = gpd.read_file(manual_file, layer=layer_key)
        # Remove any existing index_right column to avoid conflicts
        if 'index_right' in gdf.columns:
            gdf = gdf.drop(columns='index_right')
        print(f"    {layer_key}: {len(gdf)} points")
        manual_data[layer_key] = gdf
    except Exception as e:
        print(f"    Warning: Could not read layer '{layer_key}' from {manual_file}: {e}")
        manual_data[layer_key] = None

# 2. Load attractiveness points
print("\n[2] Loading attractiveness points...")
try:
    attrac_gdf = gpd.read_file(attrac_file, layer='lpg_points')
    if 'index_right' in attrac_gdf.columns:
        attrac_gdf = attrac_gdf.drop(columns='index_right')
    print(f"    lpg_points: {len(attrac_gdf)} points")
    # Ensure CRS is EPSG:3857 for consistency with manual file
    if attrac_gdf.crs != "EPSG:3857":
        attrac_gdf = attrac_gdf.to_crs("EPSG:3857")
        print("    Reprojected to EPSG:3857")
except Exception as e:
    print(f"    Error: Could not load {attrac_file}: {e}")
    raise

# 3. Filter out false points from attractiveness
print("\n[3] Removing false points from attractiveness data...")
false_gdf = manual_data.get("false_points")
if false_gdf is not None and not false_gdf.empty:
    # Ensure same CRS
    if false_gdf.crs != attrac_gdf.crs:
        false_gdf = false_gdf.to_crs(attrac_gdf.crs)
    # Use only geometry to avoid column conflicts
    false_geom = false_gdf[['geometry']].copy()
    # Spatial join: keep only attractiveness points not within any false point
    sjoin = gpd.sjoin(attrac_gdf, false_geom, how='left', predicate='within')
    attrac_valid = sjoin[sjoin['index_right'].isna()].copy()
    attrac_valid = attrac_valid.drop(columns='index_right')
    print(f"    Removed {len(attrac_gdf) - len(attrac_valid)} points that lie inside false points.")
    print(f"    Remaining attractiveness points: {len(attrac_valid)}")
else:
    attrac_valid = attrac_gdf.copy()
    print("    No false_points layer found, keeping all attractiveness points.")

# 4. Prepare manual points: add attractiveness = 1, keep all columns
print("\n[4] Adding attractiveness=1 to manual points...")
def prepare_manual(gdf, name):
    if gdf is None or gdf.empty:
        return None
    gdf = gdf.copy()
    gdf['attractiveness'] = 1.0
    print(f"    {name}: {len(gdf)} points, attractiveness set to 1")
    return gdf

manual_reseller = prepare_manual(manual_data.get("reseller"), "reseller")
manual_filling = prepare_manual(manual_data.get("filling"), "filling")
manual_primary = prepare_manual(manual_data.get("primary_depot"), "primary_depot")

# 5. Build initial layers with duplicate removal using place_id (preferred) or geometry
print("\n[5] Building initial layers (before mutual exclusion)...")
def combine_and_deduplicate(dfs, sources_description):
    """Combine multiple GeoDataFrames and drop duplicates.
       Prefer place_id if available in all sources; fallback to geometry."""
    valid_dfs = [df for df in dfs if df is not None and not df.empty]
    if not valid_dfs:
        return None

    before_count = sum(len(df) for df in valid_dfs)
    combined = pd.concat(valid_dfs, ignore_index=True)

    # Check if all source DataFrames have 'place_id'
    all_have_place_id = all('place_id' in df.columns for df in valid_dfs)

    if all_have_place_id:
        # Deduplicate using place_id (fast and reliable)
        combined = combined.drop_duplicates(subset='place_id', keep='first')
        method = "place_id"
    else:
        # Fallback to geometry (WKT)
        combined['__wkt'] = combined.geometry.apply(lambda g: g.wkt)
        combined = combined.drop_duplicates(subset='__wkt', keep='first')
        combined = combined.drop(columns='__wkt')
        method = "geometry"

    after_count = len(combined)
    combined = gpd.GeoDataFrame(combined, geometry='geometry', crs=valid_dfs[0].crs)
    print(f"    {sources_description}: {before_count} points before dedup -> {after_count} points after dedup (using {method})")
    return combined

# Initial layers
depot_gdf = manual_primary  # depot only from manual
resell_gdf = combine_and_deduplicate([manual_reseller, attrac_valid], "resell (manual reseller + attractiveness)")
filling_gdf = manual_filling  # filling only from manual

# 6. Apply mutual exclusion based on place_id (priority: depot > filling > resell)
print("\n[6] Applying mutual exclusion (depot > filling > resell)...")
if depot_gdf is not None and not depot_gdf.empty:
    depot_ids = set(depot_gdf['place_id'])
    if filling_gdf is not None and not filling_gdf.empty:
        before_fill = len(filling_gdf)
        filling_gdf = filling_gdf[~filling_gdf['place_id'].isin(depot_ids)]
        print(f"    Removed {before_fill - len(filling_gdf)} filling points that are also in depot.")
    if resell_gdf is not None and not resell_gdf.empty:
        before_resell = len(resell_gdf)
        resell_gdf = resell_gdf[~resell_gdf['place_id'].isin(depot_ids)]
        print(f"    Removed {before_resell - len(resell_gdf)} resell points that are also in depot.")

if filling_gdf is not None and not filling_gdf.empty:
    filling_ids = set(filling_gdf['place_id'])
    if resell_gdf is not None and not resell_gdf.empty:
        before_resell = len(resell_gdf)
        resell_gdf = resell_gdf[~resell_gdf['place_id'].isin(filling_ids)]
        print(f"    Removed {before_resell - len(resell_gdf)} resell points that are also in filling.")

# 7. Build resell_and_filling from the final resell and filling layers
print("\n[7] Building resell_and_filling layer...")
resell_fill_gdf = combine_and_deduplicate([resell_gdf, filling_gdf], "resell_and_filling (final resell + final filling)")

# 8. Write output GeoPackage
print(f"\n[8] Saving to {output_file}...")
if resell_gdf is not None and not resell_gdf.empty:
    resell_gdf.to_file(output_file, layer="resell", driver="GPKG")
if filling_gdf is not None and not filling_gdf.empty:
    filling_gdf.to_file(output_file, layer="filling", driver="GPKG")
if resell_fill_gdf is not None and not resell_fill_gdf.empty:
    resell_fill_gdf.to_file(output_file, layer="resell_and_filling", driver="GPKG")
if depot_gdf is not None and not depot_gdf.empty:
    depot_gdf.to_file(output_file, layer="depot", driver="GPKG")

# 9. Summary of final counts
print("\n" + "=" * 60)
print("Summary of final layers (after mutual exclusion):")
print("=" * 60)
print(f"  - depot             : {len(depot_gdf) if depot_gdf is not None else 0} points")
print(f"  - filling           : {len(filling_gdf) if filling_gdf is not None else 0} points")
print(f"  - resell            : {len(resell_gdf) if resell_gdf is not None else 0} points")
print(f"  - resell_and_filling: {len(resell_fill_gdf) if resell_fill_gdf is not None else 0} points")

# 10. Total unique points across all layers (using place_id if possible)
all_points = []
for gdf in [depot_gdf, filling_gdf, resell_gdf, resell_fill_gdf]:
    if gdf is not None and not gdf.empty:
        all_points.append(gdf)

if all_points:
    combined_all = pd.concat(all_points, ignore_index=True)
    if 'place_id' in combined_all.columns:
        total_unique = combined_all['place_id'].nunique()
        print(f"\nTotal unique points across all layers (by place_id): {total_unique}")
    else:
        combined_all['__wkt'] = combined_all.geometry.apply(lambda g: g.wkt)
        total_unique = combined_all['__wkt'].nunique()
        print(f"\nTotal unique points across all layers (by geometry): {total_unique}")
else:
    print("\nNo layers created.")

print("\nDone.")


[1] Loading manual layers...
    reseller: 89 points
    filling: 376 points
    primary_depot: 14 points
    false_points: 187 points

[2] Loading attractiveness points...
    lpg_points: 2833 points

[3] Removing false points from attractiveness data...
    Removed 108 points that lie inside false points.
    Remaining attractiveness points: 2725

[4] Adding attractiveness=1 to manual points...
    reseller: 89 points, attractiveness set to 1
    filling: 376 points, attractiveness set to 1
    primary_depot: 14 points, attractiveness set to 1

[5] Building initial layers (before mutual exclusion)...
    resell (manual reseller + attractiveness): 2814 points before dedup -> 2749 points after dedup (using place_id)

[6] Applying mutual exclusion (depot > filling > resell)...
    Removed 0 filling points that are also in depot.
    Removed 10 resell points that are also in depot.
    Removed 323 resell points that are also in filling.

[7] Building resell_and_filling layer...
    rese